# Nemotron 3.5 ASR Streaming 0.6B on OpenSLR 104 test: language prompts

This notebook runs [`nvidia/nemotron-3.5-asr-streaming-0.6b`](https://huggingface.co/nvidia/nemotron-3.5-asr-streaming-0.6b) zero-shot on the OpenSLR 104 Hinglish test set (2,942 clips, 4.76 h) with a fixed language prompt instead of `language="auto"`. The auto runs are in `nemotron-3.5-asr-streaming-0.6b.ipynb`. Together they show which prompt suits Hinglish best. It runs in Google Colab on an L4 GPU.

- **Prompts:** `hi-IN` and `en-US`.
- **Latencies:** 560 ms (`num_lookahead_tokens=6`) and 1,120 ms (`num_lookahead_tokens=13`), the same as the auto runs. A whole-clip pass applies the same attention limit as live streaming.
- **Decoding:** `transformers` defaults (greedy RNN-T decoding), float32 (the checkpoint's dtype), whole clips.
- **Inputs:** the project folder on Google Drive, and `openslr104_test.zip` (the clips in `data/processed/openslr104/test/`).
- **Outputs:** each clip's raw output in `results/nemotron-3.5-asr-streaming-0.6b-zeroshot-<prompt>-<latency>ms/hyps.jsonl`. Scores come from `notebooks/analysis/rescore-zeroshot.ipynb`.

## Setup

Nemotron 3.5 ASR needs `transformers` 5.13 or newer. The install upgrades it only if Colab's version is older, and runs before anything imports `transformers`. If Colab asks to restart the session after the install, restart and continue from the next cell.

In [ ]:
!pip install -q "transformers>=5.13.0" jiwer

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT = "/content/drive/MyDrive/projects/asr"
ZIP_PATH = f"{PROJECT}/openslr104_test.zip"
AUDIO_DIR = "/content/openslr104_test"

In [ ]:
import os
import sys

import torch

os.chdir(PROJECT)
sys.path.insert(0, f"{PROJECT}/src")
print(torch.cuda.get_device_name(0))

In [ ]:
import zipfile

from tqdm.auto import tqdm

with zipfile.ZipFile(ZIP_PATH) as z:
    for member in tqdm(z.infolist(), desc="unzip"):
        z.extract(member, AUDIO_DIR)

## Test manifest

The manifest's `audio_path` values are local paths on the machine that made the data, so each clip is found by file name in the unzipped folder.

In [ ]:
import json
from pathlib import Path

wavs = {p.name: str(p) for p in Path(AUDIO_DIR).rglob("*.wav")}
with open("data/processed/openslr104/test_manifest.jsonl", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

missing = [r["utt_id"] for r in rows if Path(r["audio_path"]).name not in wavs]
assert not missing, f"{len(missing)} clips missing, e.g. {missing[:3]}"

paths = [wavs[Path(r["audio_path"]).name] for r in rows]
refs = [r["text"] for r in rows]
hours = sum(r["duration"] for r in rows) / 3600
print(f"{len(rows)} clips in manifest, {len(wavs)} wav files, {hours:.2f} h")
print(refs[0])
print(paths[0])

## Audio

In [ ]:
import soundfile as sf

SR = 16000

audios = []
for path in tqdm(paths, desc="read audio"):
    audio, sr = sf.read(path, dtype="float32")
    assert sr == SR and audio.ndim == 1, (path, sr, audio.shape)
    audios.append(audio)

## Model

`processor.set_num_lookahead_tokens` sets the latency, and `language` sets the prompt; the processor passes both to `generate` with the inputs. The text is decoded once without special tokens for scoring, and once with them to read any language tag the model writes.

In [ ]:
import re

from transformers import AutoModelForRNNT, AutoProcessor

MODEL = "nvidia/nemotron-3.5-asr-streaming-0.6b"
BATCH = 32
LANGUAGES = ("hi-IN", "en-US")

processor = AutoProcessor.from_pretrained(MODEL)
model = AutoModelForRNNT.from_pretrained(MODEL).to("cuda").eval()
print(model.dtype, processor.supported_streaming_latencies_ms)
print({lang: processor.prompt_dictionary[lang] for lang in LANGUAGES + ("auto",)})


def transcribe(audios, language):
    inputs = processor(audios, sampling_rate=SR, language=language, return_tensors="pt")
    inputs = inputs.to(model.device, dtype=model.dtype)
    with torch.inference_mode():
        out = model.generate(**inputs, return_dict_in_generate=True)
    texts = processor.batch_decode(out.sequences, skip_special_tokens=True)
    raw = processor.batch_decode(out.sequences, skip_special_tokens=False)
    langs = [(re.findall(r"<([a-z]{2}-[A-Z]{2})>", r) or [""])[-1] for r in raw]
    return [t.strip() for t in texts], langs

A check on 5 clips at 560 ms with each prompt before the full run. Any language tag the model writes is shown in brackets.

In [ ]:
processor.set_num_lookahead_tokens(6)
for language in LANGUAGES:
    texts, langs = transcribe(audios[:5], language)
    print(f"=== {language}")
    for ref, text, lang in zip(refs, texts, langs):
        print("REF:", ref)
        print(f"HYP ({lang}):", text, "\n")

## Transcribe the test set with each prompt at both latencies

In [ ]:
import time
from collections import Counter

runs = {}
for language in LANGUAGES:
    for lookahead in (6, 13):
        processor.set_num_lookahead_tokens(lookahead)
        ms = processor.streaming_latency_ms
        texts, langs = [], []
        start = time.time()
        for b in tqdm(range(0, len(audios), BATCH), desc=f"{language} {ms} ms"):
            t, l = transcribe(audios[b:b + BATCH], language)
            texts += t
            langs += l
        elapsed = time.time() - start
        runs[language, ms] = (texts, langs)

        print(f"{language} {ms} ms: {len(texts)} outputs, {sum(not t for t in texts)} empty")
        print(f"{elapsed / 60:.1f} min, real-time factor {elapsed / (hours * 3600):.4f}")
        print("language tags:", Counter(langs).most_common(), "\n")

## Save outputs

Each clip's raw output is saved. Scoring happens in `notebooks/analysis/rescore-zeroshot.ipynb`, which reads these files, so this notebook writes no rows to `results/summary.csv`.

In [ ]:
for (language, ms), (texts, langs) in runs.items():
    out_dir = Path(f"results/nemotron-3.5-asr-streaming-0.6b-zeroshot-{language}-{ms}ms")
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / "hyps.jsonl", "w", encoding="utf-8") as f:
        for r, text, lang in zip(rows, texts, langs):
            f.write(json.dumps({"utt_id": r["utt_id"], "ref": r["text"], "hyp": text, "language": lang}, ensure_ascii=False) + "\n")
    print(out_dir)